In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [37]:
# Load datasets
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

# Standardize Team Names
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Gujarat Lions':'Gujarat Titans',
    'Kings XI Punjab':'Punjab Kings'
}

def standardize_team_names(df, columns):
    for col in columns:
        df[col] = df[col].replace(team_mapping)
    return df

# Preprocess Matches Data
matches.drop(columns=['season', 'city', 'date', 'toss_winner', 'toss_decision', 'result',
                      'result_margin', 'umpire1', 'umpire2', 'player_of_match', 'method',
                      'team1', 'team2'], inplace=True)

matches = standardize_team_names(matches, ['winner'])
# Standardize Venues
venue_mapping = {
    "M Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M Chinnaswamy Stadium, Bengaluru": "M Chinnaswamy Stadium",

    "Punjab Cricket Association Stadium, Mohali": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium, Mohali": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh": "Punjab Cricket Association Stadium, Mohali",

    "Wankhede Stadium": "Wankhede Stadium",
    "Wankhede Stadium, Mumbai": "Wankhede Stadium",

    "Eden Gardens": "Eden Gardens",
    "Eden Gardens, Kolkata": "Eden Gardens",

    "Sawai Mansingh Stadium": "Sawai Mansingh Stadium",
    "Sawai Mansingh Stadium, Jaipur": "Sawai Mansingh Stadium",

    "Rajiv Gandhi International Stadium, Uppal": "Rajiv Gandhi International Stadium, Hyderabad",
    "Rajiv Gandhi International Stadium, Uppal, Hyderabad": "Rajiv Gandhi International Stadium, Hyderabad",
    "Rajiv Gandhi International Stadium": "Rajiv Gandhi International Stadium, Hyderabad",

    "MA Chidambaram Stadium, Chepauk": "MA Chidambaram Stadium, Chepauk",
    "MA Chidambaram Stadium, Chepauk, Chennai": "MA Chidambaram Stadium, Chepauk",
    "MA Chidambaram Stadium": "MA Chidambaram Stadium, Chepauk",

    "Dr DY Patil Sports Academy": "Dr DY Patil Sports Academy",
    "Dr DY Patil Sports Academy, Mumbai": "Dr DY Patil Sports Academy",

    "Brabourne Stadium": "Brabourne Stadium",
    "Brabourne Stadium, Mumbai": "Brabourne Stadium",

    "Himachal Pradesh Cricket Association Stadium": "Himachal Pradesh Cricket Association Stadium",
    "Himachal Pradesh Cricket Association Stadium, Dharamsala": "Himachal Pradesh Cricket Association Stadium",

    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",

    "Subrata Roy Sahara Stadium": "Subrata Roy Sahara Stadium",

    "Maharashtra Cricket Association Stadium": "Maharashtra Cricket Association Stadium",
    "Maharashtra Cricket Association Stadium, Pune": "Maharashtra Cricket Association Stadium",


    "Arun Jaitley Stadium": "Arun Jaitley Stadium, Delhi",
    "Arun Jaitley Stadium, Delhi": "Arun Jaitley Stadium, Delhi",
    "Feroz Shah Kotla":"Arun Jaitley Stadium, Delhi",
}
matches['venue_canonical'] = matches['venue'].replace(venue_mapping)
matches.drop(columns=['venue'], inplace=True)

# Preprocess Deliveries Data
deliveries = deliveries[['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'total_runs', 'is_wicket']]
deliveries = standardize_team_names(deliveries, ['batting_team', 'bowling_team'])

deliveries['cum_runs'] = deliveries.groupby(['match_id', 'inning'])['total_runs'].cumsum()
deliveries['cum_wickets'] = deliveries.groupby(['match_id', 'inning'])['is_wicket'].cumsum()
deliveries['overs_completed'] = deliveries['over'] + deliveries['ball'] / 6
deliveries['current_run_rate'] = deliveries['cum_runs'] / deliveries['overs_completed'].replace(0, np.nan)

deliveries.fillna(0, inplace=True)

# Merge Preprocessed Data
merged_data = pd.merge(deliveries, matches, left_on='match_id',right_on='id', how='inner')

# Compute Target and Required Run Rate
first_innings_scores = merged_data[merged_data['inning'] == 1].groupby('match_id')['cum_runs'].max().reset_index()
first_innings_scores['target'] = first_innings_scores['cum_runs'] + 1

merged_data = merged_data.merge(first_innings_scores[['match_id', 'target']], on='match_id', how='left')
merged_data['remaining_overs'] = 20 - merged_data['overs_completed']
merged_data['required_run_rate'] = (merged_data['target'] - merged_data['cum_runs']) / merged_data['remaining_overs'].replace(0, np.nan)
merged_data.fillna(0, inplace=True)

# Create Binary Target Variable
merged_data['win'] = (merged_data['batting_team'] == merged_data['winner']).astype(int)

# Filter only Second Innings data
final_data = merged_data[merged_data['inning'] == 2][['inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
                                                       'required_run_rate', 'target', 'batting_team', 'bowling_team',
                                                       'venue_canonical', 'win']]

# Encode Categorical Features
label_encoders = {}
for col in ['batting_team', 'bowling_team', 'venue_canonical']:
    le = LabelEncoder()
    final_data[col] = le.fit_transform(final_data[col])
    label_encoders[col] = le



In [38]:
#Saving Encoder
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder

# Define categorical columns to encode
categorical_columns = ['batting_team', 'bowling_team', 'venue_canonical']

# Create a dictionary to store encoders
encoders = {}

# Apply Label Encoding and store encoders
for col in categorical_columns:
    le = LabelEncoder()
    merged_data[col + "_encoded"] = le.fit_transform(merged_data[col])
    encoders[col] = le

    # Print the mapping of encoded values
    print(f"Encoding for {col}:")
    for class_index, class_label in enumerate(le.classes_):
        print(f"{class_label} -> {class_index}")
    print("\n")

# Save encoders for later use
joblib.dump(encoders, "encoders.pkl")

# Save the DataFrame after encoding
merged_data.to_csv("encoded_data.csv", index=False)

print("Encoders saved successfully as 'encoders.pkl'.")
print("Encoded dataset saved as 'encoded_data.csv'.")


Encoding for batting_team:
Chennai Super Kings -> 0
Delhi Capitals -> 1
Gujarat Titans -> 2
Kochi Tuskers Kerala -> 3
Kolkata Knight Riders -> 4
Lucknow Super Giants -> 5
Mumbai Indians -> 6
Pune Warriors -> 7
Punjab Kings -> 8
Rajasthan Royals -> 9
Rising Pune Supergiants -> 10
Royal Challengers Bengaluru -> 11
Sunrisers Hyderabad -> 12


Encoding for bowling_team:
Chennai Super Kings -> 0
Delhi Capitals -> 1
Gujarat Titans -> 2
Kochi Tuskers Kerala -> 3
Kolkata Knight Riders -> 4
Lucknow Super Giants -> 5
Mumbai Indians -> 6
Pune Warriors -> 7
Punjab Kings -> 8
Rajasthan Royals -> 9
Rising Pune Supergiants -> 10
Royal Challengers Bengaluru -> 11
Sunrisers Hyderabad -> 12


Encoding for venue_canonical:
Arun Jaitley Stadium, Delhi -> 0
Barabati Stadium -> 1
Barsapara Cricket Stadium, Guwahati -> 2
Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow -> 3
Brabourne Stadium -> 4
Buffalo Park -> 5
De Beers Diamond Oval -> 6
Dr DY Patil Sports Academy -> 7
Dr. Y.S. Rajase

In [39]:
# Train-Test Split
X = final_data.drop(columns=['win'])
y = final_data['win']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Baseline Models
models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    print(f"\nModel: {name}")
    print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
    print(f"Test Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
    print("Classification Report:\n", classification_report(y_test, y_pred_test))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_test))


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Model: Logistic Regression
Train Accuracy: 0.7754
Test Accuracy: 0.7725
Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.74      0.76     12033
           1       0.77      0.80      0.79     13116

    accuracy                           0.77     25149
   macro avg       0.77      0.77      0.77     25149
weighted avg       0.77      0.77      0.77     25149

Confusion Matrix:
 [[ 8959  3074]
 [ 2648 10468]]

Model: Random Forest
Train Accuracy: 0.9999
Test Accuracy: 0.9974
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     12033
           1       1.00      1.00      1.00     13116

    accuracy                           1.00     25149
   macro avg       1.00      1.00      1.00     25149
weighted avg       1.00      1.00      1.00     25149

Confusion Matrix:
 [[11995    38]
 [   28 13088]]


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [13:19:33] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Model: XGBoost
Train Accuracy: 0.9965
Test Accuracy: 0.9948
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99     12033
           1       0.99      1.00      0.99     13116

    accuracy                           0.99     25149
   macro avg       0.99      0.99      0.99     25149
weighted avg       0.99      0.99      0.99     25149

Confusion Matrix:
 [[11955    78]
 [   54 13062]]
